# Memory module test

In [1]:
from pathlib import Path

from agentz.memory import init_memory

mem_root = Path("../agent/memories").resolve()
engine, SessionLocal = init_memory(str(mem_root))
print("mem_root:", mem_root)


mem_root: C:\Users\Luca\Desktop\thesis\agent\memories


In [2]:
import io
import pandas as pd
from PIL import Image

from agentz.memory import new_episode, new_observation, new_step
from agentz.memory.schema import Episode, Observation, Step, Blob

# Minimal fused_df sample
fused_df = pd.DataFrame(
    [
        {
            "content": "Settings",
            "type": "text",
            "role": "text",
            "a11y_role": "push-button",
            "a11y_is_interactive": True,
            "a11y_enabled": True,
            "a11y_visible": True,
            "score": 1.0,
        },
        {
            "content": "Firefox",
            "type": "icon",
            "role": "icon",
            "a11y_role": None,
            "a11y_is_interactive": False,
            "a11y_enabled": None,
            "a11y_visible": None,
            "score": 0.5,
        },
    ]
)

# Dummy PNG screenshot bytes
img = Image.new("RGB", (64, 64), color=(120, 180, 200))
buf = io.BytesIO()
img.save(buf, format="PNG")
screenshot_png = buf.getvalue()

a11y_xml = "<desktop-frame><application name=\"dummy\"/></desktop-frame>"

system_info = {
    "raw": {
        "os": {"pretty_name": "Ubuntu 22.04"},
        "desktop_environment": "GNOME",
        "display_server": "x11",
    }
}

task = {"id": "demo", "instruction": "Open Settings"}

with SessionLocal() as session:
    episode = new_episode(task, system_info, planner_version="v0", action_space_version="v0")
    session.add(episode)
    session.commit()

    obs1 = new_observation(
        session,
        str(mem_root),
        episode.episode_id,
        fused_df,
        system_summary="Test system",
        perception_version="v0",
        screenshot_png=screenshot_png,
        a11y_xml=a11y_xml,
        screen_w=64,
        screen_h=64,
    )
    session.commit()

    step = new_step(
        episode.episode_id,
        t=0,
        obs_before_id=obs1.obs_id,
        action_json={"type": "CLICK", "target_ui_id": "ui_0"},
    )
    session.add(step)
    session.commit()

    obs2 = new_observation(
        session,
        str(mem_root),
        episode.episode_id,
        fused_df,
        system_summary="Test system",
        perception_version="v0",
        screenshot_png=screenshot_png,
        a11y_xml=a11y_xml,
        screen_w=64,
        screen_h=64,
    )
    session.commit()

    step.obs_after_id = obs2.obs_id
    step.result_json = {"status": "ok"}
    session.commit()

    print("episodes:", session.query(Episode).count())
    print("observations:", session.query(Observation).count())
    print("steps:", session.query(Step).count())
    print("blobs:", session.query(Blob).count())


episodes: 5
observations: 12
steps: 7
blobs: 7


To reset memory, delete `agent/memories/memory.sqlite` and `agent/memories/blobs/`.

# Database inspection


In [3]:
import json
import pandas as pd
from sqlalchemy import text


In [4]:
with engine.connect() as conn:
    tables = pd.read_sql(text("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"), conn)
tables


,name
0,blobs
1,episodes
2,observations
3,skills
4,steps
5,summaries


In [5]:
with engine.connect() as conn:
    episodes = pd.read_sql(text("SELECT * FROM episodes ORDER BY started_ts_ms DESC LIMIT 5;"), conn)
episodes


,episode_id,task_id,instruction,evaluator_json,os_name,desktop_env,display_server,started_ts_ms,finished_ts_ms,final_status,final_note,planner_version,action_space_version,schema_version
0,320fc64bd5804e4398728f136264d7c0,demo,Open Settings,null,Ubuntu 22.04,GNOME,x11,1766482077731,None,ABORT,None,v0,v0,v1
1,3e4cb02432e243b0a1e57c8f05cc9c2e,dummy,Open Settings,null,unknown,unknown,unknown,1766412452477,None,DONE,rollout_index=1; selected=False; terminated_by...,v0,computer_13,v1
2,c58753214bb0474d8bcce546c8e9c92e,dummy,Open Settings,null,unknown,unknown,unknown,1766412452054,None,DONE,rollout_index=0; selected=True; terminated_by=...,v0,computer_13,v1
3,bbe92ce6792748c9b30d8191cd618c01,demo,Open Settings,null,Ubuntu 22.04,GNOME,x11,1766327704253,None,ABORT,None,v0,v0,v1
4,de33a184c28e43438402a58b0ba50983,demo,Open Settings,null,Ubuntu 22.04,GNOME,x11,1766327509229,None,ABORT,None,v0,v0,v1


In [6]:
with engine.connect() as conn:
    observations = pd.read_sql(
        text("SELECT obs_id, episode_id, ts_ms, perception_version, screenshot_blob_id, ui_parquet_blob_id, a11y_xml_blob_id FROM observations ORDER BY ts_ms DESC LIMIT 5;"),
        conn,
    )
observations


,obs_id,episode_id,ts_ms,perception_version,screenshot_blob_id,ui_parquet_blob_id,a11y_xml_blob_id
0,b84c6a9ec1644cc69f264f0a02373f30,320fc64bd5804e4398728f136264d7c0,1766482078088,v0,d8741c7cb8fcda1928ef69577be02007,0b3d1f0bd5facaebf6790566a919fcc5,510e0451c1e3e58fdc1eeeb034de8475
1,725ccdf65c304285af751c1e8037a8a3,320fc64bd5804e4398728f136264d7c0,1766482078067,v0,d8741c7cb8fcda1928ef69577be02007,0b3d1f0bd5facaebf6790566a919fcc5,510e0451c1e3e58fdc1eeeb034de8475
2,f51f08d99a61491bbf8bb3352d7a1804,3e4cb02432e243b0a1e57c8f05cc9c2e,1766412452495,v0,fedf832c77cbfe90576d8b4a24b8b4db,e9d2f357932978c2dbc1f387a3a39638,fc2e366e5b7c44548b38a52c8c2dd1cd
3,be59448dbd6d471a8dc4a86def0e3341,3e4cb02432e243b0a1e57c8f05cc9c2e,1766412452488,v0,fedf832c77cbfe90576d8b4a24b8b4db,e9d2f357932978c2dbc1f387a3a39638,fc2e366e5b7c44548b38a52c8c2dd1cd
4,9c436a161a5549a396115789bea1c2c5,3e4cb02432e243b0a1e57c8f05cc9c2e,1766412452480,v0,fedf832c77cbfe90576d8b4a24b8b4db,b572dd349d90047607405e54cda28df1,fc2e366e5b7c44548b38a52c8c2dd1cd


In [7]:
with engine.connect() as conn:
    steps = pd.read_sql(
        text("SELECT step_id, episode_id, t, obs_before_id, obs_after_id, action_json, result_json FROM steps ORDER BY t DESC LIMIT 10;"),
        conn,
    )
steps


,step_id,episode_id,t,obs_before_id,obs_after_id,action_json,result_json
0,6e3893572bb44c3e8fd98137065d1b99,c58753214bb0474d8bcce546c8e9c92e,1,8f30cb20fb1e4d2aa2c6c38f8fe89894,3c0c0faff5f74570815728b1182a3ca3,"{""type"": ""ui"", ""action"": ""DONE"", ""parameters"":...","{""status"": ""ok"", ""error"": null}"
1,0e4e019e90e843eebc42c1ba964451cb,3e4cb02432e243b0a1e57c8f05cc9c2e,1,be59448dbd6d471a8dc4a86def0e3341,f51f08d99a61491bbf8bb3352d7a1804,"{""type"": ""ui"", ""action"": ""DONE"", ""parameters"":...","{""status"": ""ok"", ""error"": null}"
2,1663e87c95a74ef2b5b4b10112093506,de33a184c28e43438402a58b0ba50983,0,a15df511f0fe4a38a88a668421def319,66154cf3c9304210b950044cad386f22,"{""type"": ""CLICK"", ""target_ui_id"": ""ui_0""}","{""status"": ""ok""}"
3,5e2dd8fb110d4eb9ad7fc2f088e1738b,bbe92ce6792748c9b30d8191cd618c01,0,9df0b0db521d4aa7964f880dcbef3915,c1f6dbe6cc8a436c83e35757ebe4297e,"{""type"": ""CLICK"", ""target_ui_id"": ""ui_0""}","{""status"": ""ok""}"
4,de1abf822f0a459ab024cb752c787a8c,c58753214bb0474d8bcce546c8e9c92e,0,648fe664123945f5991f867f4d44b33d,8f30cb20fb1e4d2aa2c6c38f8fe89894,"{""type"": ""ui"", ""action"": ""CLICK"", ""parameters""...","{""status"": ""ok"", ""error"": null}"
5,4df7f683a5fd47b8adba6a0e84b2041f,3e4cb02432e243b0a1e57c8f05cc9c2e,0,9c436a161a5549a396115789bea1c2c5,be59448dbd6d471a8dc4a86def0e3341,"{""type"": ""ui"", ""action"": ""CLICK"", ""parameters""...","{""status"": ""ok"", ""error"": null}"
6,830d80c93c1d483881fdafc671cc425b,320fc64bd5804e4398728f136264d7c0,0,725ccdf65c304285af751c1e8037a8a3,b84c6a9ec1644cc69f264f0a02373f30,"{""type"": ""CLICK"", ""target_ui_id"": ""ui_0""}","{""status"": ""ok""}"


In [8]:
with engine.connect() as conn:
    blob_counts = pd.read_sql(
        text("SELECT kind, COUNT(*) AS n FROM blobs GROUP BY kind ORDER BY n DESC;"),
        conn,
    )
blob_counts


,kind,n
0,ui_parquet,3
1,a11y_xml,2
2,screenshots,2


In [9]:
with engine.connect() as conn:
    blob_paths = pd.read_sql(
        text("SELECT blob_id, kind, uri, size_bytes, created_ts_ms FROM blobs ORDER BY created_ts_ms DESC LIMIT 10;"),
        conn,
    )
blob_paths


,blob_id,kind,uri,size_bytes,created_ts_ms
0,e9d2f357932978c2dbc1f387a3a39638,ui_parquet,file://C:\Users\Luca\Desktop\thesis\agent\memo...,3797,1766412452464
1,fc2e366e5b7c44548b38a52c8c2dd1cd,a11y_xml,file://C:\Users\Luca\Desktop\thesis\agent\memo...,31,1766412452450
2,b572dd349d90047607405e54cda28df1,ui_parquet,file://C:\Users\Luca\Desktop\thesis\agent\memo...,3813,1766412452449
3,fedf832c77cbfe90576d8b4a24b8b4db,screenshots,file://C:\Users\Luca\Desktop\thesis\agent\memo...,83,1766412452080
4,510e0451c1e3e58fdc1eeeb034de8475,a11y_xml,file://C:\Users\Luca\Desktop\thesis\agent\memo...,58,1766327509812
5,0b3d1f0bd5facaebf6790566a919fcc5,ui_parquet,file://C:\Users\Luca\Desktop\thesis\agent\memo...,4878,1766327509808
6,d8741c7cb8fcda1928ef69577be02007,screenshots,file://C:\Users\Luca\Desktop\thesis\agent\memo...,185,1766327509288


In [10]:
with engine.connect() as conn:
    digests = pd.read_sql(
        text("SELECT obs_id, ui_digest_json FROM observations ORDER BY ts_ms DESC LIMIT 3;"),
        conn,
    )
digests["ui_digest_json"] = digests["ui_digest_json"].apply(
    lambda x: x if isinstance(x, dict) else json.loads(x)
)
digests


,obs_id,ui_digest_json
0,b84c6a9ec1644cc69f264f0a02373f30,"{'items': [{'ui_id': 'ui_0', 'role': 'push-but..."
1,725ccdf65c304285af751c1e8037a8a3,"{'items': [{'ui_id': 'ui_0', 'role': 'push-but..."
2,f51f08d99a61491bbf8bb3352d7a1804,"{'items': [{'ui_id': 'ui_0', 'role': None, 'ki..."


In [11]:
if len(digests) > 0:
    last_digest = digests.iloc[0]["ui_digest_json"]
    print("keywords:", last_digest.get("keywords", [])[:20])
    print("first items:")
    print(last_digest.get("items", [])[:5])


keywords: ['settings', 'firefox']
first items:
[{'ui_id': 'ui_0', 'role': 'push-button', 'kind': 'text', 'label': 'Settings', 'enabled': True, 'visible': True, 'signature': 'push-button|settings'}, {'ui_id': 'ui_1', 'role': None, 'kind': 'icon', 'label': 'Firefox', 'enabled': None, 'visible': None, 'signature': 'none|firefox'}]
